In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/anusheshjumale/hdfc-bse-data/data/raw/gdelt/gdelt_201901.json
/kaggle/input/datasets/anusheshjumale/hdfc-bse-data/data/raw/bse/bse_20240101_20241231_page_5.json
/kaggle/input/datasets/anusheshjumale/hdfc-bse-data/data/raw/bse/bse_20190101_20191231_page_1.json
/kaggle/input/datasets/anusheshjumale/hdfc-bse-data/data/raw/bse/bse_20230101_20231231_page_1.json
/kaggle/input/datasets/anusheshjumale/hdfc-bse-data/data/raw/bse/bse_20240101_20241231_page_3.json
/kaggle/input/datasets/anusheshjumale/hdfc-bse-data/data/raw/bse/bse_20250101_20250826_page_2.json
/kaggle/input/datasets/anusheshjumale/hdfc-bse-data/data/raw/bse/bse_20220101_20221231_page_2.json
/kaggle/input/datasets/anusheshjumale/hdfc-bse-data/data/raw/bse/bse_20190101_20191231_page_4.json
/kaggle/input/datasets/anusheshjumale/hdfc-bse-data/data/raw/bse/bse_20200101_20201231_page_1.json
/kaggle/input/datasets/anusheshjumale/hdfc-bse-data/data/raw/bse/bse_20230101_20231231_page_6.json
/kaggle/input/datasets/a

# Get numerical data

In [8]:
import os
import glob
import json
import yfinance as yf
import pandas as pd
import numpy as np

BASE_DIR = "/kaggle/working"
PROCESSED_DIR = os.path.join(BASE_DIR, "data/processed")
os.makedirs(PROCESSED_DIR, exist_ok=True)

# -------------------------------------------------------------------
# 1. DOWNLOAD & SAVE MARKET OHLCV PRICES
# -------------------------------------------------------------------
print("[1/5] Downloading HDFCBANK.NS market prices...")
df_prices = yf.download("HDFCBANK.NS", start="2019-01-01", progress=False)

if isinstance(df_prices.columns, pd.MultiIndex):
    df_prices.columns = [col[0] for col in df_prices.columns]

df_prices.reset_index(inplace=True)
df_prices.rename(columns={'Date': 'date'}, inplace=True)
df_prices['date'] = pd.to_datetime(df_prices['date']).dt.tz_localize(None)
df_prices = df_prices[['date', 'Open', 'High', 'Low', 'Close', 'Volume']].dropna().sort_values('date').reset_index(drop=True)

prices_parquet_path = os.path.join(PROCESSED_DIR, "hdfc_market_prices.parquet")
df_prices.to_parquet(prices_parquet_path, index=False)
print(f"  -> Saved {len(df_prices)} trading days.")

trading_days = df_prices['date'].sort_values().unique()

# Helper: Map timestamps to next eligible trading day using 15:30 IST cutoff
def map_to_trading_day(ts):
    if pd.isna(ts):
        return None
    effective_date = ts.date() if (ts.hour < 15 or (ts.hour == 15 and ts.minute <= 30)) else ts.date() + pd.Timedelta(days=1)
    effective_ts = pd.Timestamp(effective_date)
    future_days = trading_days[trading_days >= effective_ts]
    return future_days[0] if len(future_days) > 0 else None

text_records = []

# -------------------------------------------------------------------
# 2. INGEST BSE REGULATORY FILINGS (2019-2023 JSON Files)
# -------------------------------------------------------------------
bse_pattern = "/kaggle/input/**/raw/bse/bse_*.json"
bse_files = glob.glob(bse_pattern, recursive=True)

if not bse_files:
    # Fallback to search all json files inside hdfc-bse-data
    bse_files = glob.glob("/kaggle/input/**/bse_*.json", recursive=True)

print(f"[2/5] Ingesting BSE filings from {len(bse_files)} JSON files...")
bse_count = 0

for fpath in bse_files:
    try:
        with open(fpath, 'r', encoding='utf-8') as f:
            data = json.load(f)
            
            # Handle list, dict with 'Table', or single dict
            if isinstance(data, dict):
                items = data.get('Table') or data.get('Table1') or data.get('data') or [data]
            elif isinstance(data, list):
                items = data
            else:
                items = []

            for item in items:
                if not isinstance(item, dict):
                    continue
                
                # Extract timestamp
                ts_str = (
                    item.get('NEWS_DT') or item.get('DT_TM') or item.get('DisseminationTime') or
                    item.get('timestamp') or item.get('News_dt') or item.get('dt_tm')
                )
                
                # Extract headline and description
                headline = item.get('NEWSSUB') or item.get('HEADLINE') or item.get('headline') or item.get('Subject') or ''
                desc = item.get('NEWS_BODY') or item.get('MORE') or item.get('description') or item.get('Body') or ''
                
                if ts_str and (headline or desc):
                    ts = pd.to_datetime(ts_str, errors='coerce')
                    if pd.notna(ts):
                        ts = ts.tz_localize(None) if ts.tzinfo is None else ts.tz_convert(None)
                        tdate = map_to_trading_day(ts)
                        if tdate:
                            clean_txt = f"{headline.strip()}. {desc.strip()}".strip()
                            text_records.append({'trading_date': tdate, 'clean_text': clean_txt})
                            bse_count += 1
    except Exception as e:
        continue

print(f"  -> Added {bse_count} official BSE corporate filings.")

# -------------------------------------------------------------------
# 3. INGEST BUSINESS STANDARD NEWS (2019-2026)
# -------------------------------------------------------------------
bs_matches = glob.glob("/kaggle/input/**/hdfc_business_standard_news_2019_2026.csv", recursive=True)
if bs_matches:
    bs_path = bs_matches[0]
    print(f"[3/5] Ingesting Business Standard News from: {bs_path}")
    df_bs = pd.read_csv(bs_path)
    df_bs['timestamp'] = pd.to_datetime(df_bs['timestamp']).dt.tz_localize(None)
    df_bs['trading_date'] = df_bs['timestamp'].apply(map_to_trading_day)
    df_bs['clean_text'] = df_bs['headline'].fillna('') + ". " + df_bs['description'].fillna('')
    for _, row in df_bs.dropna(subset=['trading_date']).iterrows():
        text_records.append({'trading_date': row['trading_date'], 'clean_text': row['clean_text'].strip()})
    print(f"  -> Added {len(df_bs)} Business Standard articles.")

# -------------------------------------------------------------------
# 4. INGEST KAGGLE NEWS SENTIMENT DATASET (2021-2025)
# -------------------------------------------------------------------
news_matches = glob.glob("/kaggle/input/**/HDFC_news_21.csv", recursive=True)
if news_matches:
    news_path = news_matches[0]
    print(f"[4/5] Ingesting Kaggle News dataset from: {news_path}")
    df_news = pd.read_csv(news_path)
    date_col = next((c for c in df_news.columns if 'date' in c.lower() or 'time' in c.lower()), None)
    text_col = next((c for c in df_news.columns if 'headline' in c.lower() or 'news' in c.lower() or 'title' in c.lower() or 'text' in c.lower()), None)
    
    if date_col and text_col:
        df_news[date_col] = pd.to_datetime(df_news[date_col], errors='coerce').dt.tz_localize(None)
        df_news['trading_date'] = df_news[date_col].apply(map_to_trading_day)
        for _, row in df_news.dropna(subset=['trading_date', text_col]).iterrows():
            text_records.append({'trading_date': row['trading_date'], 'clean_text': str(row[text_col]).strip()})
        print(f"  -> Added {len(df_news)} news headlines.")

# -------------------------------------------------------------------
# 5. DEDUPLICATE, MERGE & SAVE GOLD ALIGNED DATASET
# -------------------------------------------------------------------
print("[5/5] Aligning multimodal text stream to trading calendar...")
df_all_text = pd.DataFrame(text_records)
df_all_text = df_all_text.drop_duplicates(subset=['trading_date', 'clean_text']).reset_index(drop=True)

# Aggregate daily text items
df_daily_text = df_all_text.groupby('trading_date')['clean_text'].apply(lambda texts: " | ".join(texts)).reset_index()
df_daily_text.rename(columns={'trading_date': 'date', 'clean_text': 'aggregated_text'}, inplace=True)

df_aligned = pd.merge(df_prices, df_daily_text, on='date', how='left')
df_aligned['aggregated_text'] = df_aligned['aggregated_text'].fillna("No major news or filings reported for HDFC Bank on this trading day.")

gold_aligned_path = os.path.join(PROCESSED_DIR, "hdfc_phase1_aligned.parquet")
df_aligned.to_parquet(gold_aligned_path, index=False)

print("\n" + "="*60)
print("Phase 1 Gold Dataset Created Successfully!")
print(f"Total Trading Days: {len(df_aligned)}")
active_coverage = (df_aligned['aggregated_text'] != "No major news or filings reported for HDFC Bank on this trading day.").sum()
print(f"Days with active news/filings: {active_coverage} ({active_coverage / len(df_aligned) * 100:.2f}%)")
print(f"Saved to: {gold_aligned_path}")
print("="*60)

[1/5] Downloading HDFCBANK.NS market prices...


/tmp/ipykernel_58/1786616860.py:16: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df_prices = yf.download("HDFCBANK.NS", start="2019-01-01", progress=False)


  -> Saved 1896 trading days.
[2/5] Ingesting BSE filings from 31 JSON files...
  -> Added 1428 official BSE corporate filings.
[3/5] Ingesting Business Standard News from: /kaggle/input/datasets/anusheshjumale/hdfc-bs-news-dataset/hdfc_business_standard_news_2019_2026.csv
  -> Added 645 Business Standard articles.
[4/5] Ingesting Kaggle News dataset from: /kaggle/input/datasets/muhammedshaheb/hdfc-stock-news-dataset-sentiment-analysis-21-25/HDFC_news_21.csv


/tmp/ipykernel_58/1786616860.py:124: FutureWarning: Parsed string "08:07:25 29/04/2025 pm IST" included an un-recognized timezone "IST". Dropping unrecognized timezones is deprecated; in a future version this will raise. Instead pass the string without the timezone, then use .tz_localize to convert to a recognized timezone.
  df_news[date_col] = pd.to_datetime(df_news[date_col], errors='coerce').dt.tz_localize(None)
/tmp/ipykernel_58/1786616860.py:124: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_news[date_col] = pd.to_datetime(df_news[date_col], errors='coerce').dt.tz_localize(None)
/tmp/ipykernel_58/1786616860.py:124: FutureWarning: Parsed string "08:10:13 28/04/2025 pm IST" included an un-recognized timezone "IST". Dropping unrecognized timezones is deprecated; in a future version this will raise. Instead pass the string without the timezone

  -> Added 2514 news headlines.
[5/5] Aligning multimodal text stream to trading calendar...

Phase 1 Gold Dataset Created Successfully!
Total Trading Days: 1896
Days with active news/filings: 1433 (75.58%)
Saved to: /kaggle/working/data/processed/hdfc_phase1_aligned.parquet


In [9]:
import os
import json
import pandas as pd
import numpy as np

BASE_DIR = "/kaggle/working"
PROCESSED_DIR = os.path.join(BASE_DIR, "data/processed")
os.makedirs(PROCESSED_DIR, exist_ok=True)

# -------------------------------------------------------------------
# 0. STANDALONE STANDARD SCALER CLASS (Replaces sklearn.preprocessing)
# -------------------------------------------------------------------
class SimpleStandardScaler:
    def __init__(self):
        self.mean_ = None
        self.scale_ = None
        self.feature_cols = None

    def fit(self, df, feature_cols):
        self.feature_cols = feature_cols
        self.mean_ = df[feature_cols].mean(axis=0).to_dict()
        # Sample standard deviation with epsilon floor to prevent division by zero
        std = df[feature_cols].std(axis=0)
        self.scale_ = std.replace(0, 1e-8).to_dict()
        return self

    def transform(self, df):
        df_out = df.copy()
        for col in self.feature_cols:
            mean = self.mean_[col]
            std = self.scale_[col]
            df_out[col] = (df_out[col] - mean) / std
        return df_out

    def save(self, filepath):
        with open(filepath, 'w', encoding='utf-8') as f:
            json.dump({
                "mean": self.mean_,
                "scale": self.scale_,
                "feature_cols": self.feature_cols
            }, f, indent=4)

# -------------------------------------------------------------------
# 1. LOAD PHASE 1 DATASET
# -------------------------------------------------------------------
input_path = os.path.join(PROCESSED_DIR, "hdfc_phase1_aligned.parquet")
df = pd.read_parquet(input_path)
df['date'] = pd.to_datetime(df['date'])
df.sort_values('date', inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"Loaded Phase 1 dataset with {len(df)} rows.")

# -------------------------------------------------------------------
# 2. TECHNICAL INDICATORS (Pure Pandas)
# -------------------------------------------------------------------
# Exponential Moving Averages
df['ema_20'] = df['Close'].ewm(span=20, adjust=False).mean()
df['ema_50'] = df['Close'].ewm(span=50, adjust=False).mean()

# RSI (14-day)
delta = df['Close'].diff()
gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)
avg_gain = gain.ewm(alpha=1/14, adjust=False).mean()
avg_loss = loss.ewm(alpha=1/14, adjust=False).mean()
rs = avg_gain / (avg_loss + 1e-9)
df['rsi'] = 100 - (100 / (1 + rs))

# MACD (12, 26, 9)
ema_12 = df['Close'].ewm(span=12, adjust=False).mean()
ema_26 = df['Close'].ewm(span=26, adjust=False).mean()
df['macd'] = ema_12 - ema_26
df['macd_signal'] = df['macd'].ewm(span=9, adjust=False).mean()

# Stochastic Oscillator %K (14-day)
low_14 = df['Low'].rolling(14).min()
high_14 = df['High'].rolling(14).max()
df['stoch_k'] = 100 * ((df['Close'] - low_14) / (high_14 - low_14 + 1e-9))

# Bollinger Bands (20-day, 2 std)
df['bb_mid'] = df['Close'].rolling(20).mean()
bb_std = df['Close'].rolling(20).std()
df['bb_upper'] = df['bb_mid'] + (2 * bb_std)
df['bb_lower'] = df['bb_mid'] - (2 * bb_std)
df['bb_width'] = (df['bb_upper'] - df['bb_lower']) / (df['bb_mid'] + 1e-9)

# ATR (14-day)
tr1 = df['High'] - df['Low']
tr2 = (df['High'] - df['Close'].shift(1)).abs()
tr3 = (df['Low'] - df['Close'].shift(1)).abs()
tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
df['atr'] = tr.ewm(alpha=1/14, adjust=False).mean()

# On-Balance Volume (OBV)
obv_dir = np.sign(df['Close'].diff()).fillna(0)
df['obv'] = (obv_dir * df['Volume']).cumsum()

# 20-Day Rolling Daily Volatility for Dynamic Thresholding
df['daily_log_ret'] = np.log(df['Close'] / df['Close'].shift(1))
df['rolling_vol_20'] = df['daily_log_ret'].rolling(window=20).std()

# Drop early warm-up rows
warmup_drop = 50
df = df.iloc[warmup_drop:].reset_index(drop=True)
print(f"Dropped {warmup_drop} warm-up rows. Remaining rows: {len(df)}")

# -------------------------------------------------------------------
# 3. DYNAMIC VOLATILITY-BASED MULTI-HORIZON (3-DAY) LABELING
# -------------------------------------------------------------------
HORIZON = 3
df['fwd_return'] = np.log(df['Close'].shift(-HORIZON) / df['Close'])

# Dynamic threshold scaled by 3-day volatility
VOL_MULTIPLIER = 0.75
df['dynamic_threshold'] = df['rolling_vol_20'] * np.sqrt(HORIZON) * VOL_MULTIPLIER

def assign_dynamic_label(row):
    ret = row['fwd_return']
    thresh = row['dynamic_threshold']
    if pd.isna(ret) or pd.isna(thresh):
        return np.nan
    if ret > thresh:
        return 2  # Buy
    elif ret < -thresh:
        return 0  # Sell
    else:
        return 1  # Hold

df['label'] = df.apply(assign_dynamic_label, axis=1)

# Drop trailing rows lacking full forward return lookahead
df.dropna(subset=['label'], inplace=True)
df['label'] = df['label'].astype(int)

print("\n" + "="*50)
print(f"Dynamic Multi-Horizon (3-Day) Class Balance:")
print("="*50)
print(df['label'].value_counts().sort_index().rename({0: 'Sell (0)', 1: 'Hold (1)', 2: 'Buy (2)'}))

# -------------------------------------------------------------------
# 4. CHRONOLOGICAL PARTITIONING (60/20/20)
# -------------------------------------------------------------------
feature_cols = [
    'Open', 'High', 'Low', 'Close', 'Volume',
    'rsi', 'macd', 'macd_signal', 'stoch_k',
    'bb_width', 'atr', 'obv', 'ema_20', 'ema_50', 'rolling_vol_20'
]

total_len = len(df)
train_idx = int(total_len * 0.60)
val_idx = int(total_len * 0.80)

df_train = df.iloc[:train_idx]
df_val = df.iloc[train_idx:val_idx]
df_test = df.iloc[val_idx:]

print(f"\nPartition Split Sizes:")
print(f"Train: {len(df_train)} | Validation: {len(df_val)} | Test: {len(df_test)}")

# -------------------------------------------------------------------
# 5. STRICT LEAKAGE-FREE SCALING (Fitted ONLY on Train Partition)
# -------------------------------------------------------------------
scaler = SimpleStandardScaler()
scaler.fit(df_train, feature_cols)

df_train_scaled = scaler.transform(df_train)
df_val_scaled = scaler.transform(df_val)
df_test_scaled = scaler.transform(df_test)

# Save scaler parameters and partitioned Parquet files
scaler_path = os.path.join(PROCESSED_DIR, "hdfc_scaler.json")
scaler.save(scaler_path)

df_train_scaled.to_parquet(os.path.join(PROCESSED_DIR, "train_partition.parquet"), index=False)
df_val_scaled.to_parquet(os.path.join(PROCESSED_DIR, "val_partition.parquet"), index=False)
df_test_scaled.to_parquet(os.path.join(PROCESSED_DIR, "test_partition.parquet"), index=False)

print("\n[✓] Phase 2 Finished: Clean dynamic labels and scaled partitions saved successfully.")

Loaded Phase 1 dataset with 1896 rows.
Dropped 50 warm-up rows. Remaining rows: 1846

Dynamic Multi-Horizon (3-Day) Class Balance:
label
Sell (0)     378
Hold (1)    1054
Buy (2)      411
Name: count, dtype: int64

Partition Split Sizes:
Train: 1105 | Validation: 369 | Test: 369

[✓] Phase 2 Finished: Clean dynamic labels and scaled partitions saved successfully.


In [10]:
!pip install "numpy<2" scipy scikit-learn transformers --force-reinstall

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 3.2 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of scipy to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 3.6 MB/s eta 0:00:00
^C
ERROR: Operation cancelled by user


In [11]:
import os
import hashlib
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
import pandas as pd
import numpy as np
from tqdm import tqdm

BASE_DIR = "/kaggle/working"
PROCESSED_DIR = os.path.join(BASE_DIR, "data/processed")
EMBEDDING_CACHE_DIR = os.path.join(BASE_DIR, "data/processed/embedding_cache")
os.makedirs(EMBEDDING_CACHE_DIR, exist_ok=True)

# Device configuration (Utilize Kaggle GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 1. Load FinBERT Model and Tokenizer
MODEL_NAME = "ProsusAI/finbert"
print(f"Loading tokenizer and model from {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(device)
model.eval()

# 2. Load Partitions
df_train = pd.read_parquet(os.path.join(PROCESSED_DIR, "train_partition.parquet"))
df_val = pd.read_parquet(os.path.join(PROCESSED_DIR, "val_partition.parquet"))
df_test = pd.read_parquet(os.path.join(PROCESSED_DIR, "test_partition.parquet"))

def get_text_embedding(text):
    """
    Computes the [CLS] embedding for a text string using FinBERT with disk caching.
    """
    if not isinstance(text, str) or not text.strip():
        text = "No major news or filings reported for HDFC Bank on this trading day."
        
    # Hash string for cache lookup filename
    text_hash = hashlib.md5(text.encode('utf-8')).hexdigest()
    cache_file = os.path.join(EMBEDDING_CACHE_DIR, f"{text_hash}.npy")
    
    if os.path.exists(cache_file):
        return np.load(cache_file)
    
    # Tokenize and encode
    inputs = tokenizer(
        text,
        return_tensors="pt",
        max_length=512,
        truncation=True,
        padding="max_length"
    ).to(device)
    
    with torch.no_grad():
        outputs = model(**inputs)
        # Extract [CLS] token representation (shape: [1, 768])
        cls_embedding = outputs.last_hidden_state[:, 0, :].cpu().numpy().squeeze(0)
        
    # Save to disk cache
    np.save(cache_file, cls_embedding)
    return cls_embedding

def process_partition_embeddings(df, partition_name):
    print(f"\nExtracting FinBERT embeddings for {partition_name} ({len(df)} rows)...")
    embeddings = []
    
    for text in tqdm(df['aggregated_text'].values):
        emb = get_text_embedding(text)
        embeddings.append(emb)
        
    emb_array = np.array(embeddings)
    out_path = os.path.join(PROCESSED_DIR, f"{partition_name}_embeddings.npy")
    np.save(out_path, emb_array)
    print(f"[✓] Saved {partition_name} embeddings shape {emb_array.shape} to {out_path}")
    return emb_array

# Extract for all three partitions
train_emb = process_partition_embeddings(df_train, "train")
val_emb = process_partition_embeddings(df_val, "val")
test_emb = process_partition_embeddings(df_test, "test")

print("\nPhase 3 Complete: All daily text data successfully converted to FinBERT 768-d dense embeddings.")

Using device: cpu
Loading tokenizer and model from ProsusAI/finbert...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
classifier.weight            | UNEXPECTED |  | 
classifier.bias              | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Extracting FinBERT embeddings for train (1105 rows)...


100%|██████████| 1105/1105 [05:32<00:00,  3.32it/s]


[✓] Saved train embeddings shape (1105, 768) to /kaggle/working/data/processed/train_embeddings.npy

Extracting FinBERT embeddings for val (369 rows)...


100%|██████████| 369/369 [02:07<00:00,  2.88it/s]


[✓] Saved val embeddings shape (369, 768) to /kaggle/working/data/processed/val_embeddings.npy

Extracting FinBERT embeddings for test (369 rows)...


100%|██████████| 369/369 [00:25<00:00, 14.68it/s]

[✓] Saved test embeddings shape (369, 768) to /kaggle/working/data/processed/test_embeddings.npy

Phase 3 Complete: All daily text data successfully converted to FinBERT 768-d dense embeddings.


In [12]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

BASE_DIR = "/kaggle/working"
PROCESSED_DIR = os.path.join(BASE_DIR, "data/processed")
ARTIFACTS_DIR = os.path.join(BASE_DIR, "artifacts")
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

# -------------------------------------------------------------------
# 1. PURE PYTHON EVALUATION METRICS (Zero sklearn dependency)
# -------------------------------------------------------------------
def calculate_metrics(y_true, y_pred, num_classes=3):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    f1_scores = []
    class_precisions = []
    class_recalls = []
    
    for c in range(num_classes):
        tp = np.sum((y_pred == c) & (y_true == c))
        fp = np.sum((y_pred == c) & (y_true != c))
        fn = np.sum((y_pred != c) & (y_true == c))
        
        precision = tp / (tp + fp + 1e-9)
        recall = tp / (tp + fn + 1e-9)
        f1 = 2 * (precision * recall) / (precision + recall + 1e-9)
        
        class_precisions.append(precision)
        class_recalls.append(recall)
        f1_scores.append(f1)
        
    macro_f1 = np.mean(f1_scores)
    accuracy = np.mean(y_true == y_pred)
    return {
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "precision": class_precisions,
        "recall": class_recalls,
        "f1": f1_scores
    }

# -------------------------------------------------------------------
# 2. MULTI-CLASS FOCAL LOSS
# -------------------------------------------------------------------
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.alpha = alpha  # Tensor of class weights [3]
        self.gamma = gamma

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(logits, targets, reduction='none', weight=self.alpha)
        pt = torch.exp(-ce_loss)
        focal_loss = ((1.0 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()

# -------------------------------------------------------------------
# 3. GATED MULTIMODAL ARCHITECTURE
# -------------------------------------------------------------------
class GatedMultimodalStockPredictor(nn.Module):
    def __init__(self, num_features=15, hidden_dim=96, lstm_layers=2, dropout=0.4):
        super(GatedMultimodalStockPredictor, self).__init__()
        
        # 1. Numerical Branch: Bidirectional LSTM
        self.lstm = nn.LSTM(
            input_size=num_features,
            hidden_size=hidden_dim,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if lstm_layers > 1 else 0.0
        )
        lstm_out_dim = hidden_dim * 2
        
        # 2. Text Branch: FinBERT Projection
        self.text_proj = nn.Sequential(
            nn.Linear(768, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, lstm_out_dim)
        )
        
        # 3. Gating Layer: Evaluates whether to inject text context
        self.gate_layer = nn.Sequential(
            nn.Linear(lstm_out_dim * 2, lstm_out_dim),
            nn.Sigmoid()
        )
        
        # 4. Final Classification Head
        self.classifier = nn.Sequential(
            nn.Linear(lstm_out_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 3)
        )

    def forward(self, x_num, x_text):
        # Numerical LSTM Forward Pass
        lstm_out, (hn, _) = self.lstm(x_num) # [Batch, 30, lstm_out_dim]
        
        # Text Projection Forward Pass
        proj_text = self.text_proj(x_text)   # [Batch, lstm_out_dim]
        text_key = proj_text.unsqueeze(1)    # [Batch, 1, lstm_out_dim]
        
        # Attention Mechanism over the 30-day timeline
        scores = torch.sum(lstm_out * text_key, dim=-1, keepdim=True) / (lstm_out.size(-1) ** 0.5)
        attn_weights = F.softmax(scores, dim=1) # [Batch, 30, 1]
        context_vec = torch.sum(attn_weights * lstm_out, dim=1) # [Batch, lstm_out_dim]
        
        # Pooled LSTM representation from final forward/backward states
        lstm_pooled = torch.cat([hn[-2], hn[-1]], dim=1) # [Batch, lstm_out_dim]
        
        # Gating Mechanism
        gate = self.gate_layer(torch.cat([lstm_pooled, context_vec], dim=1))
        gated_context = gate * context_vec
        
        # Fusion & Classification
        fused = torch.cat([lstm_pooled, gated_context], dim=1)
        logits = self.classifier(fused)
        
        return logits, attn_weights.squeeze(-1)

# -------------------------------------------------------------------
# 4. DATASET & DATALOADERS
# -------------------------------------------------------------------
class HDFCDataset(Dataset):
    def __init__(self, df_scaled, embeddings, feature_cols, lookback=30):
        self.features = df_scaled[feature_cols].values.astype(np.float32)
        self.labels = df_scaled['label'].values.astype(np.int64)
        self.embeddings = embeddings.astype(np.float32)
        self.lookback = lookback
        self.valid_indices = range(lookback - 1, len(self.features))

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        end_idx = self.valid_indices[idx]
        start_idx = end_idx - self.lookback + 1
        x_num = self.features[start_idx:end_idx + 1]
        x_text = self.embeddings[end_idx]
        y = self.labels[end_idx]
        return torch.tensor(x_num), torch.tensor(x_text), torch.tensor(y)

feature_cols = [
    'Open', 'High', 'Low', 'Close', 'Volume',
    'rsi', 'macd', 'macd_signal', 'stoch_k',
    'bb_width', 'atr', 'obv', 'ema_20', 'ema_50', 'rolling_vol_20'
]

df_train = pd.read_parquet(os.path.join(PROCESSED_DIR, "train_partition.parquet"))
df_val = pd.read_parquet(os.path.join(PROCESSED_DIR, "val_partition.parquet"))
df_test = pd.read_parquet(os.path.join(PROCESSED_DIR, "test_partition.parquet"))

train_emb = np.load(os.path.join(PROCESSED_DIR, "train_embeddings.npy"))
val_emb = np.load(os.path.join(PROCESSED_DIR, "val_embeddings.npy"))
test_emb = np.load(os.path.join(PROCESSED_DIR, "test_embeddings.npy"))

train_dataset = HDFCDataset(df_train, train_emb, feature_cols, lookback=30)
val_dataset = HDFCDataset(df_val, val_emb, feature_cols, lookback=30)
test_dataset = HDFCDataset(df_test, test_emb, feature_cols, lookback=30)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# -------------------------------------------------------------------
# 5. CLASS WEIGHTS & FOCAL LOSS INITIALIZATION
# -------------------------------------------------------------------
train_labels = [train_dataset[i][2].item() for i in range(len(train_dataset))]
counts = np.bincount(train_labels)
alpha_weights = len(train_labels) / (len(counts) * counts)
alpha_tensor = torch.tensor(alpha_weights, dtype=torch.float32).to(device)

model = GatedMultimodalStockPredictor(num_features=len(feature_cols)).to(device)
criterion = FocalLoss(alpha=alpha_tensor, gamma=2.0)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=25)

# -------------------------------------------------------------------
# 6. TRAINING LOOP WITH VALIDATION MACRO-F1 CHECKPOINTING
# -------------------------------------------------------------------
EPOCHS = 25
best_val_f1 = 0.0
best_model_path = os.path.join(ARTIFACTS_DIR, "best_gated_focal_model.pth")

print("\nStarting Training with Gated Attention & Focal Loss...")
for epoch in range(EPOCHS):
    model.train()
    total_train_loss = 0.0
    
    for x_num, x_text, y in train_loader:
        x_num, x_text, y = x_num.to(device), x_text.to(device), y.to(device)
        
        optimizer.zero_grad()
        logits, _ = model(x_num, x_text)
        loss = criterion(logits, y)
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_train_loss += loss.item()
        
    scheduler.step()
    
    # Validation
    model.eval()
    val_preds, val_targets = [], []
    total_val_loss = 0.0
    
    with torch.no_grad():
        for x_num, x_text, y in val_loader:
            x_num, x_text, y = x_num.to(device), x_text.to(device), y.to(device)
            logits, _ = model(x_num, x_text)
            loss = criterion(logits, y)
            total_val_loss += loss.item()
            
            preds = torch.argmax(logits, dim=1)
            val_preds.extend(preds.cpu().numpy())
            val_targets.extend(y.cpu().numpy())
            
    val_metrics = calculate_metrics(val_targets, val_preds)
    val_f1 = val_metrics["macro_f1"]
    
    train_loss_avg = total_train_loss / len(train_loader)
    val_loss_avg = total_val_loss / len(val_loader)
    
    print(f"Epoch {epoch+1:02d}/{EPOCHS} | Train: {train_loss_avg:.4f} | Val: {val_loss_avg:.4f} | Val Macro-F1: {val_f1:.4f} | Acc: {val_metrics['accuracy']:.4f}")
    
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), best_model_path)
        print(f"  -> Checkpoint Saved! Best Val Macro-F1: {best_val_f1:.4f}")

print(f"\nTraining Complete. Best Model Saved to: {best_model_path}")

Training on device: cpu

Starting Training with Gated Attention & Focal Loss...
Epoch 01/25 | Train: 0.5916 | Val: 0.5090 | Val Macro-F1: 0.2494 | Acc: 0.2647
  -> Checkpoint Saved! Best Val Macro-F1: 0.2494
Epoch 02/25 | Train: 0.5607 | Val: 0.5126 | Val Macro-F1: 0.2281 | Acc: 0.2441
Epoch 03/25 | Train: 0.5350 | Val: 0.5341 | Val Macro-F1: 0.1965 | Acc: 0.2176
Epoch 04/25 | Train: 0.5361 | Val: 0.5521 | Val Macro-F1: 0.2174 | Acc: 0.2324
Epoch 05/25 | Train: 0.5193 | Val: 0.6151 | Val Macro-F1: 0.2099 | Acc: 0.2353
Epoch 06/25 | Train: 0.5198 | Val: 0.5798 | Val Macro-F1: 0.2369 | Acc: 0.2471
Epoch 07/25 | Train: 0.5054 | Val: 0.6069 | Val Macro-F1: 0.2120 | Acc: 0.2294
Epoch 08/25 | Train: 0.4913 | Val: 0.5608 | Val Macro-F1: 0.2254 | Acc: 0.2353
Epoch 09/25 | Train: 0.4850 | Val: 0.5927 | Val Macro-F1: 0.2424 | Acc: 0.2500
Epoch 10/25 | Train: 0.4828 | Val: 0.6051 | Val Macro-F1: 0.2889 | Acc: 0.2882
  -> Checkpoint Saved! Best Val Macro-F1: 0.2889
Epoch 11/25 | Train: 0.4711 | Va

In [13]:
import os
import torch
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt

BASE_DIR = "/kaggle/working"
PROCESSED_DIR = os.path.join(BASE_DIR, "data/processed")
ARTIFACTS_DIR = os.path.join(BASE_DIR, "artifacts")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Load Test Dataset & Best Model
df_test = pd.read_parquet(os.path.join(PROCESSED_DIR, "test_partition.parquet"))
test_emb = np.load(os.path.join(PROCESSED_DIR, "test_embeddings.npy"))

feature_cols = [
    'Open', 'High', 'Low', 'Close', 'Volume',
    'rsi', 'macd', 'macd_signal', 'stoch_k',
    'bb_width', 'atr', 'obv', 'ema_20', 'ema_50', 'rolling_vol_20'
]

# (Assuming HDFCDataset is still in memory from the previous cell)
test_dataset = HDFCDataset(df_test, test_emb, feature_cols, lookback=30)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Initialize model and load best Focal/Gated checkpoint
model = GatedMultimodalStockPredictor(num_features=len(feature_cols)).to(device)
best_model_path = os.path.join(ARTIFACTS_DIR, "best_gated_focal_model.pth")
model.load_state_dict(torch.load(best_model_path))
model.eval()

# 2. Run Inference
all_preds, all_targets = [], []

with torch.no_grad():
    for x_num, x_text, y in test_loader:
        x_num, x_text, y = x_num.to(device), x_text.to(device), y.to(device)
        logits, _ = model(x_num, x_text)
        preds = torch.argmax(logits, dim=1)
        
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(y.cpu().numpy())

# 3. Performance Metrics
print("\n" + "="*50)
print("TEST SET CLASSIFICATION REPORT (3-Day Horizon)")
print("="*50)
print(classification_report(all_targets, all_preds, target_names=['Sell (0)', 'Hold (1)', 'Buy (2)']))

# 4. Multi-Horizon Backtest Simulation
# Map labels: 2 (Buy) -> Long (+1), 0 (Sell) -> Short (-1), 1 (Hold) -> Flat (0)
positions = np.array([1 if p == 2 else (-1 if p == 0 else 0) for p in all_preds])

# Align test prices with the test dataset valid indices
# The fwd_return here is already computed as the 3-day forward return in Phase 2
test_fwd_returns = df_test['fwd_return'].values[29:]

# Strategy return = position * 3-day forward return
strategy_returns = positions * test_fwd_returns[:len(positions)]

# Transaction cost simulation (5 basis points per position change)
# Because we are simulating a 3-day holding period implicitly, we charge costs when position changes
trades = np.abs(np.diff(positions, prepend=0))
transaction_costs = trades * 0.0005
net_strategy_returns = strategy_returns - transaction_costs

cum_market_returns = np.cumsum(test_fwd_returns[:len(positions)])
cum_strategy_gross = np.cumsum(strategy_returns)
cum_strategy_net = np.cumsum(net_strategy_returns)

print(f"\nBacktest Results (Test Set - {len(positions)} Trading Windows):")
print(f"Cumulative Market Return (Passive): {cum_market_returns[-1]*100:.2f}%")
print(f"Cumulative Strategy Return (Gross): {cum_strategy_gross[-1]*100:.2f}%")
print(f"Cumulative Strategy Return (Net of Costs): {cum_strategy_net[-1]*100:.2f}%")


TEST SET CLASSIFICATION REPORT (3-Day Horizon)
              precision    recall  f1-score   support

    Sell (0)       0.23      0.63      0.33        78
    Hold (1)       0.59      0.33      0.43       206
     Buy (2)       0.29      0.04      0.06        56

    accuracy                           0.35       340
   macro avg       0.37      0.33      0.27       340
weighted avg       0.46      0.35      0.35       340


Backtest Results (Test Set - 340 Trading Windows):
Cumulative Market Return (Passive): -79.40%
Cumulative Strategy Return (Gross): 34.18%
Cumulative Strategy Return (Net of Costs): 33.73%
